In [3]:
import sys; sys.path.append("..")
import pandas as pd
from collections import Counter
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from src.data import load_data
from src.config import RANDOM_STATE
from src.experiment_log import ExperimentLogger

X_train, _, y_train, _, labels = load_data()
lg = ExperimentLogger("thai")

def under_cap(y, cap=5000):          # giam lop lon xuong toi da 5000 mau
    return {k: cap for k, v in Counter(y).items() if v > cap}

def smote_to(y, target=2000):        # nang lop hiem len 2000 mau
    return {k: target for k, v in Counter(y).items() if v < target}

def rf_factory():
    return RandomForestClassifier(n_estimators=100, max_depth=10, max_features="log2",
                                  n_jobs=-1, random_state=RANDOM_STATE)

def hgb_factory(**kw):
    return HistGradientBoostingClassifier(early_stopping=False,
                                          random_state=RANDOM_STATE, **kw)   # thay bang tham so hgb_tuned neu co

def make_variants(factory):
    us = lambda: RandomUnderSampler(sampling_strategy=under_cap, random_state=RANDOM_STATE)
    sm = lambda: SMOTE(sampling_strategy=smote_to, k_neighbors=3, random_state=RANDOM_STATE)
    return {
        "N2_under":       ImbPipeline([("us", us()), ("clf", factory())]),
        "N3_smote":       ImbPipeline([("sc", StandardScaler()), ("sm", sm()), ("clf", factory())]),
        "N4_under_smote": ImbPipeline([("us", us()), ("sc", StandardScaler()), ("sm", sm()), ("clf", factory())]),
    }

# N0 va N1 cho RF: lay tu log cua B (rf_d10_cw_None, rf_d10_cw_balanced), khong chay lai
# N0 cho HGB: hgb_tuned (da co tu Buoc 4); N1 cho HGB:
lg.run(hgb_factory(class_weight="balanced"), X_train, y_train, "hgb_N1_balanced",
       strategy="class_weight=balanced", notes="N1 tren HGB")

for base, factory in [("rf", rf_factory), ("hgb", hgb_factory)]:
    for vname, pipe in make_variants(factory).items():
        lg.run(pipe, X_train, y_train, f"{base}_{vname}",
               strategy=f"{vname} (cap=5000 / SMOTE target=2000, k=3)",
               notes="thi nghiem mat can bang")

In [4]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import precision_recall_fscore_support

def per_class_report(model, X, y):
    cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
    pred = cross_val_predict(model, X, y, cv=cv)
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=range(12), zero_division=0)
    return pd.DataFrame({"precision": p, "recall": r, "f1": f}, index=labels).round(3)

rep = pd.concat({
    "N0": per_class_report(rf_factory(), X_train, y_train),
    "N3_smote": per_class_report(make_variants(rf_factory)["N3_smote"], X_train, y_train),
}, axis=1)
rep.to_csv("../results/cv/imbalance_per_class_rf.csv")
rep

N0                N3_smote              
                           precision recall     f1 precision recall     f1
ARP_poisioning                 0.981  0.991  0.986     0.982  0.990  0.986
DDOS_Slowloris                 0.983  0.969  0.976     0.887  0.979  0.931
DOS_SYN_Hping                  1.000  1.000  1.000     1.000  1.000  1.000
MQTT_Publish                   1.000  0.998  0.999     1.000  0.998  0.999
Metasploit_Brute_Force_SSH     0.880  0.759  0.815     0.759  0.759  0.759
NMAP_FIN_SCAN                  0.947  0.818  0.878     0.720  0.818  0.766
NMAP_OS_DETECTION              1.000  1.000  1.000     1.000  1.000  1.000
NMAP_TCP_scan                  0.999  0.998  0.998     0.999  0.998  0.998
NMAP_UDP_SCAN                  0.994  0.986  0.990     0.994  0.963  0.978
NMAP_XMAS_TREE_SCAN            1.000  0.998  0.999     1.000  0.998  0.999
Thing_Speak                    0.988  0.986  0.987     0.988  0.985  0.986
Wipro_bulb                     0.963  0.897  0.929     0.933  0.960  0.946

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn.ensemble import HistGradientBoostingClassifier
from src.config import RANDOM_STATE


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
splits = list(skf.split(X_train, y_train))
train_idx, val_idx = splits[0]

X_tr = X_train.iloc[train_idx]
y_tr = y_train.iloc[train_idx]
X_va = X_train.iloc[val_idx]
y_va = y_train.iloc[val_idx]

learning_rates = [0.1, 0.03]
l2_values = [0.0, 0.1, 1.0]


for lr in learning_rates:
    for l2 in l2_values:
        model = HistGradientBoostingClassifier(
            learning_rate=lr,
            l2_regularization=l2,
            early_stopping=False,
            random_state=RANDOM_STATE
        )
        model.fit(X_tr, y_tr)

       
        proba = model.predict_proba(X_va)
        pred = np.argmax(proba, axis=1)

    
        macro_f1 = f1_score(y_va, pred, average="macro")
        acc = accuracy_score(y_va, pred)
        has_nan = bool(np.isnan(proba).any())
        num_predicted_classes = len(np.unique(pred))

        print(
            f"lr={lr} l2={l2}: "
            f"macroF1={macro_f1:.3f} "
            f"acc={acc:.3f} "
            f"NaN={has_nan} "
            f"so_lop_du_doan={num_predicted_classes}"
        )

lr=0.1 l2=0.0: macroF1=0.415 acc=0.816 NaN=False so_lop_du_doan=11
lr=0.1 l2=0.1: macroF1=0.972 acc=0.998 NaN=False so_lop_du_doan=12
lr=0.1 l2=1.0: macroF1=0.968 acc=0.998 NaN=False so_lop_du_doan=12
lr=0.03 l2=0.0: macroF1=0.964 acc=0.998 NaN=False so_lop_du_doan=12
lr=0.03 l2=0.1: macroF1=0.968 acc=0.998 NaN=False so_lop_du_doan=12
lr=0.03 l2=1.0: macroF1=0.966 acc=0.998 NaN=False so_lop_du_doan=12


In [6]:
from sklearn.ensemble import HistGradientBoostingClassifier
from src.config import RANDOM_STATE
from src.experiment_log import ExperimentLogger


HGB_TUNED = {
    "learning_rate": 0.03,
    "l2_regularization": 0.1,
    "min_samples_leaf": 2,
}

def hgb_factory(**kw):
    
    params = {
        "early_stopping": False,
        "random_state": RANDOM_STATE,
    }
    for k, v in HGB_TUNED.items():
        params[k] = v
    for k, v in kw.items():
        params[k] = v

    return HistGradientBoostingClassifier(**params)


lg = ExperimentLogger("bao")


model_balanced = hgb_factory(class_weight="balanced")
lg.run(
    model_balanced,
    X_train,
    y_train,
    "hgbT_N1_balanced",
    strategy="N1: class_weight=balanced",
    notes="HGB tuned co dinh",
)


variants = make_variants(hgb_factory)
for vname, pipe in variants.items():
    exp_name = f"hgbT_{vname}"
    lg.run(
        pipe,
        X_train,
        y_train,
        exp_name,
        strategy=vname,
        notes="HGB tuned co dinh",
    )

In [7]:
import pandas as pd
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import precision_recall_fscore_support
from src.config import RANDOM_STATE, CV_DIR

def per_class_report(model, X, y, class_labels=labels):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    pred = cross_val_predict(model, X, y, cv=cv)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y, pred, labels=list(range(12)), zero_division=0
    )

    report_df = pd.DataFrame({
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }, index=class_labels)

    return report_df.round(3)

variants = make_variants(rf_factory)

rep_n0 = per_class_report(rf_factory(), X_train, y_train)
rep_under = per_class_report(variants["N2_under"], X_train, y_train)
rep_smote = per_class_report(variants["N3_smote"], X_train, y_train)

rep = pd.concat(
    {
        "N0": rep_n0,
        "N2_under": rep_under,
        "N3_smote": rep_smote,
    },
    axis=1
)

# Lưu kết quả ra thư mục CV_DIR
output_csv = CV_DIR / "imbalance_per_class_rf.csv"
rep.to_csv(output_csv)

rep

N0                N2_under                \
                           precision recall     f1 precision recall     f1   
ARP_poisioning                 0.981  0.991  0.986     0.983  0.992  0.988   
DDOS_Slowloris                 0.983  0.969  0.976     0.983  0.977  0.980   
DOS_SYN_Hping                  1.000  1.000  1.000     1.000  1.000  1.000   
MQTT_Publish                   1.000  0.998  0.999     1.000  0.998  0.999   
Metasploit_Brute_Force_SSH     0.880  0.759  0.815     0.880  0.759  0.815   
NMAP_FIN_SCAN                  0.947  0.818  0.878     0.947  0.818  0.878   
NMAP_OS_DETECTION              1.000  1.000  1.000     1.000  1.000  1.000   
NMAP_TCP_scan                  0.999  0.998  0.998     1.000  0.998  0.999   
NMAP_UDP_SCAN                  0.994  0.986  0.990     0.995  0.986  0.990   
NMAP_XMAS_TREE_SCAN            1.000  0.998  0.999     1.000  0.998  0.999   
Thing_Speak                    0.988  0.986  0.987     0.990  0.989  0.989   
Wipro_bulb                     0.963  0.897  0.929     0.964  0.914  0.938   

                            N3_smote                
                           precision recall     f1  
ARP_poisioning                 0.982  0.990  0.986  
DDOS_Slowloris                 0.887  0.979  0.931  
DOS_SYN_Hping                  1.000  1.000  1.000  
MQTT_Publish                   1.000  0.998  0.999  
Metasploit_Brute_Force_SSH     0.759  0.759  0.759  
NMAP_FIN_SCAN                  0.720  0.818  0.766  
NMAP_OS_DETECTION              1.000  1.000  1.000  
NMAP_TCP_scan                  0.999  0.998  0.998  
NMAP_UDP_SCAN                  0.994  0.963  0.978  
NMAP_XMAS_TREE_SCAN            1.000  0.998  0.999  
Thing_Speak                    0.988  0.985  0.986  
Wipro_bulb                     0.933  0.960  0.946

In [8]:
import pandas as pd
from src.config import CV_DIR

def paired_check(base_name, names):
    base_df = pd.read_csv(CV_DIR / f"{base_name}.csv")
    base_scores = base_df["macro_f1"]

    for name in names:
        var_df = pd.read_csv(CV_DIR / f"{name}.csv")
        var_scores = var_df["macro_f1"]
        diff = var_scores - base_scores

        n_folds = len(diff)
        mean_diff = diff.mean()
        std_diff = diff.std()

        if std_diff > 0 and n_folds > 1:
            t_stat = mean_diff / (std_diff / (n_folds ** 0.5))
        else:
            t_stat = 0.0

        print(f"{name}: chenh TB={mean_diff:+.4f}  t={t_stat:+.2f}")

print("RF so voi N0 cua thai")
paired_check(
    "rf_d10_cw_None",
    ["rf_N2_under", "rf_N3_smote", "rf_N4_under_smote"]
)

print("HGB so voi hgb_tuned")
paired_check(
    "hgb_tuned",
    ["hgbT_N1_balanced", "hgbT_N2_under", "hgbT_N3_smote", "hgbT_N4_under_smote"]
)

RF so voi N0 cua thai
rf_N2_under: chenh TB=+0.0015  t=+1.74
rf_N3_smote: chenh TB=-0.0168  t=-4.59
rf_N4_under_smote: chenh TB=-0.0175  t=-3.25
HGB so voi hgb_tuned
hgbT_N1_balanced: chenh TB=-0.0085  t=-1.00
hgbT_N2_under: chenh TB=+0.0087  t=+1.94
hgbT_N3_smote: chenh TB=-0.0024  t=-0.73
hgbT_N4_under_smote: chenh TB=-0.0005  t=-0.13


In [ ]:
from src.experiment_log import ExperimentLogger

lg = ExperimentLogger("bao")
lg.run(hgb_factory(), X_train, y_train, "hgbT_N0",
       strategy="N0: khong xu ly", notes="kiem tra khop hgb_tuned")

,accuracy,balanced_acc,macro_f1,weighted_f1,mcc,recall_c0,recall_c1,recall_c2,recall_c3,recall_c4,...,recall_c6,recall_c7,recall_c8,recall_c9,recall_c10,recall_c11,model,fold,fit_time,pred_time
0,0.998198,0.959175,0.967838,0.998194,0.995556,0.990984,0.964706,1.0,0.998489,0.833333,...,1.0,1.000000,0.987893,1.000000,0.991837,0.942857,hgbT_N0,0,9.466846,0.234721
1,0.997721,0.929716,0.933833,0.997705,0.994381,0.990984,0.988235,1.0,1.000000,0.500000,...,1.0,0.993789,0.985472,0.996885,0.986939,0.914286,hgbT_N0,1,13.659912,0.240938
2,0.997880,0.955255,0.951105,0.997888,0.994773,0.991803,0.988372,1.0,1.000000,0.800000,...,1.0,0.993789,0.985472,0.996885,0.985306,0.971429,hgbT_N0,2,10.472802,0.255063
3,0.998569,0.983367,0.978089,0.998575,0.996471,0.992623,1.000000,1.0,0.995475,0.833333,...,1.0,1.000000,0.985507,1.000000,0.993464,1.000000,hgbT_N0,3,11.730522,0.277491
4,0.998092,0.950813,0.969548,0.998082,0.995295,0.994262,0.941176,1.0,0.998492,0.833333,...,1.0,1.000000,0.992754,0.996894,0.988562,0.914286,hgbT_N0,4,10.768051,0.244295
